# Part 1 — Dataset Selection and Preparation

**Dataset:** WMT16 English ↔ Turkish  
**Source:** HuggingFace `datasets` library (`wmt16`, config `tr-en`)

This notebook covers:
- Dataset characteristics and split sizes
- Preprocessing pipeline
- Statistical summary (sentence lengths, vocabulary)
- Example input-output pairs

In [1]:
import sys
sys.path.insert(0, '..')  # make modules importable from notebooks/

import matplotlib.pyplot as plt
import pandas as pd
import numpy as nps
from collections import Counter

from modules.dataset import load_wmt16, preprocess_pair, get_samples, dataset_stats

## 1.1 Load Dataset Splits

In [2]:
train_ds  = load_wmt16(split='train')
valid_ds  = load_wmt16(split='validation')
test_ds   = load_wmt16(split='test')

print(f"\nSplit sizes:")
print(f"  Train:      {len(train_ds):>8,}")
print(f"  Validation: {len(valid_ds):>8,}")
print(f"  Test:       {len(test_ds):>8,}")
print(f"  Total:      {len(train_ds)+len(valid_ds)+len(test_ds):>8,}")

Loading WMT16 tr-en [train] split...


README.md: 0.00B [00:00, ?B/s]

tr-en/train-00000-of-00001.parquet:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

tr-en/validation-00000-of-00001.parquet:   0%|          | 0.00/153k [00:00<?, ?B/s]

tr-en/test-00000-of-00001.parquet:   0%|          | 0.00/466k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/205756 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1001 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3000 [00:00<?, ? examples/s]

  Loaded 205756 sentence pairs.
Loading WMT16 tr-en [validation] split...
  Loaded 1001 sentence pairs.
Loading WMT16 tr-en [test] split...
  Loaded 3000 sentence pairs.

Split sizes:
  Train:       205,756
  Validation:    1,001
  Test:          3,000
  Total:       209,757


## 1.2 Dataset Characteristics

In [3]:
print("Dataset schema:")
print(train_ds.features)
print("\nExample raw entry:")
print(train_ds[0])

Dataset schema:
{'translation': Translation(languages=['tr', 'en'])}

Example raw entry:
{'translation': {'en': "Kosovo's privatisation process is under scrutiny", 'tr': "Kosova'nın özelleştirme süreci büyüteç altında"}}


In [4]:
# Compute stats for each split
splits = {'train': train_ds, 'validation': valid_ds, 'test': test_ds}
stats = {name: dataset_stats(ds) for name, ds in splits.items()}

stats_df = pd.DataFrame(stats).T
stats_df.index.name = 'split'
print("\nDataset Statistics:")
print(stats_df.to_string())


Dataset Statistics:
                size  avg_en_tokens  avg_tr_tokens  max_en_tokens  max_tr_tokens  min_en_tokens  min_tr_tokens
split                                                                                                         
train       205756.0           21.4           17.6          136.0          121.0            1.0            1.0
validation    1001.0           19.5           14.0          119.0           96.0            1.0            1.0
test          3000.0           19.5           14.7          101.0           60.0            1.0            1.0


## 1.3 Sentence Length Distributions

In [ ]:
# Sample 5000 pairs from train for plotting
import os
import random
import numpy as np
random.seed(42)
sample_idx = random.sample(range(len(train_ds)), 5000)
sample = [train_ds[i] for i in sample_idx]

en_lens = [len(s['translation']['en'].split()) for s in sample]
tr_lens = [len(s['translation']['tr'].split()) for s in sample]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(en_lens, bins=50, color='steelblue', alpha=0.8, edgecolor='white')
axes[0].set_title('English Sentence Length (tokens)', fontsize=13)
axes[0].set_xlabel('Word count')
axes[0].set_ylabel('Frequency')
axes[0].axvline(np.mean(en_lens), color='red', linestyle='--', label=f'Mean: {np.mean(en_lens):.1f}')
axes[0].legend()

axes[1].hist(tr_lens, bins=50, color='darkorange', alpha=0.8, edgecolor='white')
axes[1].set_title('Turkish Sentence Length (tokens)', fontsize=13)
axes[1].set_xlabel('Word count')
axes[1].set_ylabel('Frequency')
axes[1].axvline(np.mean(tr_lens), color='red', linestyle='--', label=f'Mean: {np.mean(tr_lens):.1f}')
axes[1].legend()

plt.suptitle('WMT16 English-Turkish: Sentence Length Distribution (5K sample)', fontsize=14, y=1.02)
plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/length_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved → results/length_distribution.png')

## 1.4 Preprocessing Pipeline

In [ ]:
# Demonstrate the preprocessing pipeline
print("Preprocessing pipeline demo:\n")

raw_examples = [
    ('  Hello ,   world!  ', '  Merhaba ,  dünya!  '),
    ('The   EU Parliament\tcalled for action.', 'AB  Parlamentosu  harekete  geçilmesini istedi.'),
]

for raw_en, raw_tr in raw_examples:
    clean_en, clean_tr = preprocess_pair(raw_en, raw_tr)
    print(f"  Raw EN : {repr(raw_en)}")
    print(f"  Clean  : {repr(clean_en)}")
    print(f"  Raw TR : {repr(raw_tr)}")
    print(f"  Clean  : {repr(clean_tr)}")
    print()

In [ ]:
# Filter empty pairs check
empty_count = sum(
    1 for item in train_ds
    if not item['translation']['en'].strip() or not item['translation']['tr'].strip()
)
print(f"Empty pairs in train split: {empty_count} / {len(train_ds)} ({100*empty_count/len(train_ds):.3f}%)")

## 1.5 Example Input-Output Pairs

In [ ]:
en_samples, tr_samples = get_samples(test_ds, n=10, seed=42)

print("10 random test set pairs (EN → TR):\n")
print(f"{'#':<4} {'English':<60} {'Turkish'}")
print("-" * 120)
for i, (en, tr) in enumerate(zip(en_samples, tr_samples), 1):
    en_disp = en[:58] + '..' if len(en) > 60 else en
    tr_disp = tr[:55] + '..' if len(tr) > 57 else tr
    print(f"{i:<4} {en_disp:<60} {tr_disp}")

## 1.6 Summary

| | |
|---|---|
| **Dataset** | WMT16 English ↔ Turkish (`wmt16`, `tr-en`) |
| **Train pairs** | ~207,000 |
| **Validation pairs** | ~1,001 |
| **Test pairs** | ~3,000 |
| **Domain** | News (Setimes corpus — news from South-East Europe) |
| **Preprocessing** | Strip whitespace, collapse multiple spaces, filter empty pairs |
| **Evaluation subset** | 200 random test samples (seed=42) |
| **RAG corpus** | Full train split (up to 50K pairs for speed) |